# 📊 Credit Risk Loan Default Analysis

## 1. Project Introduction

This exploratory data analysis (EDA) project studies applicant and loan characteristics to understand how they are associated with loan defaults. The target column is `loan_status` (0 = no default, 1 = default). The work includes data understanding, data cleaning, univariate and bivariate analysis, multivariate analysis, statistical testing, and business-oriented findings.

**Note:** Place the dataset file `credit_risk_dataset.csv` in the same folder as this notebook before running it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Dataset Understanding

In [ ]:
# Keep credit_risk_dataset.csv in the same folder as this notebook.
df = pd.read_csv("credit_risk_dataset.csv")

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data Cleaning & Preparation

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated()]

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df['person_emp_length'].describe()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x='person_emp_length',
    bins=15,
    color='#4C78A8',
    edgecolor='white',
    alpha=0.85
)

plt.title('Distribution of Employment Length', fontsize=16, fontweight='bold')
plt.xlabel('Employment Length (Years)', fontsize=12)
plt.ylabel('Number of People', fontsize=12)

sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
df['loan_int_rate'].describe()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x='loan_int_rate',
    bins=20,
    color='#CDB4DB',       # pastel lavender 💜
    edgecolor='white',
    alpha=0.9
)

plt.title(
    'Distribution of Loan Interest Rate',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Interest Rate (%)', fontsize=12)
plt.ylabel('Number of Loans', fontsize=12)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df['person_emp_length'].median()

In [ ]:
# Fill missing loan_int_rate

df['loan_int_rate'] = df['loan_int_rate'].fillna(
    df['loan_int_rate'].median()
)

In [ ]:
# Verify that missing values are gone

df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Identify numerical columns

df.select_dtypes(include=np.number).columns

## 4. Univariate Analysis

Explore the distribution of individual numerical and categorical variables. Use the plots and summary statistics to understand patterns and identify unusual values.

In [ ]:
numerical_columns = [
    'person_age',
    'person_income',
    'person_emp_length',
    'loan_amnt',
    'loan_int_rate',
    'loan_percent_income',
    'cb_person_cred_hist_length'
]

# Readable names for graphs
column_names = {
    'person_age': 'Applicant Age',
    'person_income': 'Applicant Income',
    'person_emp_length': 'Employment Length',
    'loan_amnt': 'Loan Amount',
    'loan_int_rate': 'Loan Interest Rate',
    'loan_percent_income': 'Loan-to-Income Ratio',
    'cb_person_cred_hist_length': 'Credit History Length'
}

for column in numerical_columns:
    
    plt.figure(figsize=(10, 5))
    
    sns.boxplot(
        x=df[column],
        color='#4C78A8',
        width=0.5,
        fliersize=5,
        linewidth=1.5
    )
    
    plt.title(
        f'Boxplot of {column_names[column]}',
        fontsize=16,
        fontweight='bold'
    )
    
    plt.xlabel(column_names[column], fontsize=12)
    
    sns.despine()
    plt.tight_layout()
    plt.show()

In [ ]:
# Detect outliers using the IQR method
# Calculate IQR for person_income

Q1 = df['person_income'].quantile(0.25)
Q3 = df['person_income'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

In [ ]:
# Count the outliers

outliers_income = df[
    (df['person_income'] < lower_bound) |
    (df['person_income'] > upper_bound)
]

print("Number of potential income outliers:", len(outliers_income))

In [ ]:
numerical_columns = [
    'person_age',
    'person_income',
    'person_emp_length',
    'loan_amnt',
    'loan_int_rate',
    'loan_percent_income',
    'cb_person_cred_hist_length'
]

for column in numerical_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print(f"{column}: {len(outliers)} potential outliers")

In [ ]:
df[df['person_age'] > 100][['person_age', 'person_income', 'loan_amnt', 'loan_status']]

In [ ]:
df[df['person_emp_length'] > 50][
    ['person_age', 'person_emp_length', 'person_income', 'loan_status']
]

In [ ]:
df = df[(df['person_age'] >= 18) & (df['person_age'] <= 100)]

In [ ]:
df[df['person_emp_length'] > df['person_age']][
    ['person_age', 'person_emp_length']
]

In [ ]:
df.loc[
    df['person_emp_length'] > df['person_age'],
    'person_emp_length'
] = np.nan

In [ ]:
df['person_emp_length'] = df['person_emp_length'].fillna(
    df['person_emp_length'].median()
)

In [ ]:
print("Maximum age:", df['person_age'].max())
print("Maximum employment length:", df['person_emp_length'].max())

In [ ]:
categorical_columns = [
    'person_home_ownership',
    'loan_intent',
    'loan_grade',
    'cb_person_default_on_file'
]

In [ ]:
for column in categorical_columns:
    print("\n", column)
    print(df[column].value_counts())

In [ ]:
df['loan_status'].value_counts()

In [ ]:
df['loan_status'].value_counts(normalize=True) * 100

In [ ]:
plt.figure(figsize=(10, 5))

ax = sns.countplot(
    data=df,
    x='loan_status',
    hue='loan_status',
    palette=['#A8DADC', '#FFAFCC'],
    legend=False
)

plt.title(
    'Loan Default Distribution',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Status', fontsize=12)
plt.ylabel('Number of Applicants', fontsize=12)

plt.xticks([0, 1], ['No Default', 'Default'])

# Add count labels
for container in ax.containers:
    ax.bar_label(
        container,
        fontsize=10,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

### 💡 Key Insights

* Most borrowers have successfully repaid/are not classified as default.
* A smaller proportion of loans are classified as defaults.
* This indicates that the dataset is somewhat imbalanced, so default rate should be considered alongside the overall loan count.

In [ ]:
default_rate = df['loan_status'].mean() * 100

print(f"Default Rate: {default_rate:.2f}%")

In [ ]:
grade_default = (
    df.groupby('loan_grade')['loan_status']
    .agg(['count', 'sum', 'mean'])
)

grade_default.columns = [
    'Total Loans',
    'Total Defaults',
    'Default Rate'
]

grade_default['Default Rate'] = (
    grade_default['Default Rate'] * 100
).round(2)

grade_default

In [ ]:
plt.figure(figsize=(10, 5))

grade_data = grade_default.reset_index()

colors = [
    '#A8DADC',  # pastel blue
    '#BDE0FE',  # light blue
    '#CDB4DB',  # pastel purple
    '#FFC8DD',  # pastel pink
    '#FFAFCC',  # soft pink
    '#FFD6A5',  # pastel peach
    '#CAFFBF'   # pastel green
]

ax = sns.barplot(
    data=grade_data,
    x='loan_grade',
    y='Default Rate',
    hue='loan_grade',
    palette=colors[:len(grade_data)],
    legend=False
)

plt.title(
    'Default Rate by Loan Grade',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Grade', fontsize=12)
plt.ylabel('Default Rate (%)', fontsize=12)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=10,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

### 💡 Key Insights

* Default rate varies across loan grades.
* Lower-quality loan grades generally represent borrowers with higher observed credit risk.
* Loan grade can therefore be considered an important variable when analyzing the likelihood of loan default.

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=df,
    x='loan_status',
    y='person_age',
    hue='loan_status',
    palette=['#A8DADC', '#FFAFCC'],  # pastel blue 💙 + soft pink 🩷
    legend=False,
    width=0.5,
    linewidth=1.5
)

plt.title(
    'Applicant Age vs Loan Default',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Status', fontsize=12)
plt.ylabel('Age (Years)', fontsize=12)

plt.xticks([0, 1], ['No Default', 'Default'])

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df['person_income'].describe()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x='person_income',
    bins=50,
    kde=True,
    color='#FFD6A5',       # pastel peach 
    edgecolor='white',
    alpha=0.9,
    line_kws={'color': '#CDB4DB', 'linewidth': 2}
)

plt.title(
    'Distribution of Applicant Income',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Annual Income', fontsize=12)
plt.ylabel('Number of Applicants', fontsize=12)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df['loan_amnt'].describe()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x='loan_amnt',
    bins=30,
    kde=True,
    color='#CDB4DB',       # pastel lavender 💜
    edgecolor='white',
    alpha=0.9,
    line_kws={
        'color': '#FFAFCC',  # soft pink 🩷
        'linewidth': 2
    }
)

plt.title(
    'Distribution of Loan Amount',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Amount', fontsize=12)
plt.ylabel('Number of Applicants', fontsize=12)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=df,
    x='loan_status',
    y='loan_amnt',
    hue='loan_status',
    palette=['#A8DADC', '#FFAFCC'],  # pastel blue 💙 + soft pink 🩷
    legend=False,
    width=0.5,
    linewidth=1.5,
    fliersize=5
)

plt.title(
    'Loan Amount vs Loan Default',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Status', fontsize=12)
plt.ylabel('Loan Amount', fontsize=12)

plt.xticks([0, 1], ['No Default', 'Default'])

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df['loan_int_rate'].describe()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x='loan_int_rate',
    bins=30,
    kde=True,
    color='#CDB4DB',       # pastel lavender 💜
    edgecolor='white',
    alpha=0.9,
    line_kws={
        'color': '#FFAFCC',  # soft pink 🩷
        'linewidth': 2
    }
)

plt.title(
    'Distribution of Loan Interest Rate',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Interest Rate (%)', fontsize=12)
plt.ylabel('Number of Loans', fontsize=12)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=df,
    x='loan_status',
    y='loan_int_rate',
    hue='loan_status',
    palette=['#A8DADC', '#FFAFCC'],  # pastel blue 💙 + soft pink 🩷
    legend=False,
    width=0.5,
    linewidth=1.5,
    fliersize=5
)

plt.title(
    'Interest Rate vs Loan Default',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Status', fontsize=12)
plt.ylabel('Interest Rate (%)', fontsize=12)

plt.xticks([0, 1], ['No Default', 'Default'])

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df['loan_percent_income'].describe()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x='loan_percent_income',
    bins=30,
    kde=True,
    color='#CAFFBF',       # pastel green 💚
    edgecolor='white',
    alpha=0.9,
    line_kws={
        'color': '#FFAFCC',  # soft pink 🩷
        'linewidth': 2
    }
)

plt.title(
    'Distribution of Loan-to-Income Ratio',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Percent Income', fontsize=12)
plt.ylabel('Number of Applicants', fontsize=12)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=df,
    x='loan_status',
    y='loan_percent_income',
    hue='loan_status',
    palette=['#A8DADC', '#FFAFCC'],  # pastel blue 💙 + soft pink 🩷
    legend=False,
    width=0.5,
    linewidth=1.5,
    fliersize=5
)

plt.title(
    'Loan-to-Income Ratio vs Loan Default',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Status', fontsize=12)
plt.ylabel('Loan Percent Income', fontsize=12)

plt.xticks([0, 1], ['No Default', 'Default'])

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df['person_emp_length'].describe()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x='person_emp_length',
    bins=20,
    kde=True,
    color='#BDE0FE',       # pastel light blue 🩵
    edgecolor='white',
    alpha=0.9,
    line_kws={
        'color': '#FFAFCC',  # soft pink 🩷
        'linewidth': 2
    }
)

plt.title(
    'Distribution of Employment Length',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Employment Length (Years)', fontsize=12)
plt.ylabel('Number of Applicants', fontsize=12)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=df,
    x='loan_status',
    y='person_emp_length',
    hue='loan_status',
    palette=['#A8DADC', '#FFAFCC'],  # pastel blue 💙 + soft pink 🩷
    legend=False,
    width=0.5,
    linewidth=1.5,
    fliersize=5
)

plt.title(
    'Employment Length vs Loan Default',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Status', fontsize=12)
plt.ylabel('Employment Length (Years)', fontsize=12)

plt.xticks([0, 1], ['No Default', 'Default'])

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df['person_home_ownership'].value_counts()

In [ ]:
plt.figure(figsize=(10, 5))

ax = sns.countplot(
    data=df,
    x='person_home_ownership',
    hue='person_home_ownership',
    palette=[
        '#A8DADC',  # pastel blue 💙
        '#CDB4DB',  # pastel lavender 💜
        '#FFC8DD',  # pastel pink 🌸
        '#CAFFBF'   # pastel green 💚
    ],
    legend=False
)

plt.title(
    'Distribution of Home Ownership',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Home Ownership', fontsize=12)
plt.ylabel('Number of Applicants', fontsize=12)

# Add count labels
for container in ax.containers:
    ax.bar_label(
        container,
        fontsize=10,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

ax = sns.countplot(
    data=df,
    x='person_home_ownership',
    hue='loan_status',
    palette=['#A8DADC', '#FFAFCC']  # pastel blue 💙 + soft pink 🩷
)

plt.title(
    'Loan Default by Home Ownership',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Home Ownership', fontsize=12)
plt.ylabel('Number of Applicants', fontsize=12)

plt.legend(
    title='Loan Status',
    labels=['No Default', 'Default']
)

# Add count labels
for container in ax.containers:
    ax.bar_label(
        container,
        fontsize=9,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('person_home_ownership')['loan_status'].mean() * 100

In [ ]:
df['loan_intent'].value_counts()

In [ ]:
plt.figure(figsize=(10, 5))

ax = sns.countplot(
    data=df,
    x='loan_intent',
    palette='Set2'
)

plt.title('Distribution of Loan Intent', fontsize=16, fontweight='bold')
plt.xlabel('Loan Purpose', fontsize=12)
plt.ylabel('Number of Applicants', fontsize=12)

plt.xticks(rotation=45, ha='right')

# Add count labels on top of bars
for container in ax.containers:
    ax.bar_label(container, fontsize=10, padding=3)

# Remove unnecessary borders
sns.despine()

plt.tight_layout()
plt.show()

### 💡 Key Insights

* Default rates differ across different loan purposes.
* Some loan intents show a comparatively higher proportion of defaults than others.
* This suggests that the purpose for which a borrower takes a loan may be associated with repayment behavior.

In [ ]:
plt.figure(figsize=(10, 5))

ax = sns.countplot(
    data=df,
    x='loan_intent',
    hue='loan_status',
    palette=['#A8DADC', '#FFAFCC'],  # pastel blue 💙 + soft pink 🩷
)

plt.title(
    'Loan Default by Loan Purpose',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Purpose', fontsize=12)
plt.ylabel('Number of Applicants', fontsize=12)

plt.xticks(rotation=45)

plt.legend(
    title='Loan Status',
    labels=['No Default', 'Default']
)

# Add count labels
for container in ax.containers:
    ax.bar_label(
        container,
        fontsize=8,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('loan_intent')['loan_status'].mean().sort_values(ascending=False) * 100

In [ ]:
df['loan_grade'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(10, 5))

grade_order = sorted(df['loan_grade'].unique())

ax = sns.countplot(
    data=df,
    x='loan_grade',
    order=grade_order,
    hue='loan_grade',
    palette=[
        '#A8DADC',  # pastel blue 💙
        '#BDE0FE',  # light blue 🩵
        '#CDB4DB',  # pastel lavender 💜
        '#FFC8DD',  # pastel pink 🌸
        '#FFAFCC',  # soft pink 🩷
        '#FFD6A5',  # pastel peach 🍑
        '#CAFFBF'   # pastel green 💚
    ][:len(grade_order)],
    legend=False
)

plt.title(
    'Distribution of Loan Grades',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Grade', fontsize=12)
plt.ylabel('Number of Loans', fontsize=12)

# Add count labels
for container in ax.containers:
    ax.bar_label(
        container,
        fontsize=10,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('loan_grade')['loan_status'].mean().sort_values(ascending=False) * 100

In [ ]:
df['cb_person_default_on_file'].value_counts()

In [ ]:
plt.figure(figsize=(8, 5))

ax = sns.countplot(
    data=df,
    x='cb_person_default_on_file',
    hue='cb_person_default_on_file',
    palette=[
        '#A8DADC',  # pastel blue 💙
        '#FFAFCC'   # soft pink 🩷
    ],
    legend=False
)

plt.title(
    'Previous Default History',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Previous Default on File', fontsize=12)
plt.ylabel('Number of Applicants', fontsize=12)

# Add count labels
for container in ax.containers:
    ax.bar_label(
        container,
        fontsize=10,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

ax = sns.countplot(
    data=df,
    x='cb_person_default_on_file',
    hue='loan_status',
    palette=['#A8DADC', '#FFAFCC']  # pastel blue 💙 + soft pink 🩷
)

plt.title(
    'Current Loan Default by Previous Default History',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Previous Default on File', fontsize=12)
plt.ylabel('Number of Applicants', fontsize=12)

plt.legend(
    title='Current Loan Status',
    labels=['No Default', 'Default']
)

# Add count labels
for container in ax.containers:
    ax.bar_label(
        container,
        fontsize=9,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('cb_person_default_on_file')['loan_status'].mean() * 100

## 5. Bivariate Analysis

In [ ]:
df.groupby('loan_status')['person_income'].mean()

In [ ]:
df.groupby('loan_status')['person_income'].median()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x='loan_status',
    y='person_income'
)

plt.title('Applicant Income vs Loan Default')
plt.xlabel('Loan Status')
plt.ylabel('Annual Income')

plt.xticks([0, 1], ['No Default', 'Default'])

plt.show()

In [ ]:
df.groupby('loan_status')['loan_amnt'].agg(
    ['mean', 'median', 'min', 'max']
)

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=df,
    x='loan_status',
    y='loan_amnt',
    hue='loan_status',
    palette=['#A8DADC', '#FFAFCC'],  # pastel blue 💙 + soft pink 🩷
    legend=False,
    width=0.5,
    linewidth=1.5,
    fliersize=5
)

plt.title(
    'Loan Amount vs Loan Default',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Status', fontsize=12)
plt.ylabel('Loan Amount', fontsize=12)

plt.xticks([0, 1], ['No Default', 'Default'])

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('loan_status')['loan_int_rate'].agg(
    ['mean', 'median', 'min', 'max']
)

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=df,
    x='loan_status',
    y='loan_int_rate',
    hue='loan_status',
    palette=['#A8DADC', '#FFAFCC'],  # pastel blue 💙 + soft pink 🩷
    legend=False,
    width=0.5,
    linewidth=1.5,
    fliersize=5
)

plt.title(
    'Interest Rate vs Loan Default',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Status', fontsize=12)
plt.ylabel('Interest Rate (%)', fontsize=12)

plt.xticks([0, 1], ['No Default', 'Default'])

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('loan_status')['loan_percent_income'].agg(
    ['mean', 'median', 'min', 'max']
)

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=df,
    x='loan_status',
    y='loan_percent_income',
    hue='loan_status',
    palette=['#A8DADC', '#FFAFCC'],  # pastel blue 💙 + soft pink 🩷
    legend=False,
    width=0.5,
    linewidth=1.5,
    fliersize=5
)

plt.title(
    'Loan-to-Income Ratio vs Loan Default',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Status', fontsize=12)
plt.ylabel('Loan Percent Income', fontsize=12)

plt.xticks([0, 1], ['No Default', 'Default'])

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
home_default_rate = (
    df.groupby('person_home_ownership')['loan_status']
    .mean()
    .mul(100)
)

print(home_default_rate)

In [ ]:
plt.figure(figsize=(10, 5))

colors = [
    '#A8DADC',  # pastel blue 💙
    '#BDE0FE',  # light blue 🩵
    '#CDB4DB',  # pastel lavender 💜
    '#FFC8DD'   # pastel pink 🌸
]

ax = sns.barplot(
    x=home_default_rate.index,
    y=home_default_rate.values,
    hue=home_default_rate.index,
    palette=colors[:len(home_default_rate)],
    legend=False
)

plt.title(
    'Default Rate by Home Ownership',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Home Ownership', fontsize=12)
plt.ylabel('Default Rate (%)', fontsize=12)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=10,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

### 💡 Key Insights

* Default rates vary among different home-ownership categories.
* Home ownership status may provide additional information about a borrower's financial situation.
* However, this variable should be analyzed together with income, employment length, loan amount, and other factors.

In [ ]:
intent_default_rate = (
    df.groupby('loan_intent')['loan_status']
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(intent_default_rate)

In [ ]:
plt.figure(figsize=(10, 5))

colors = [
    '#A8DADC',  # pastel blue 💙
    '#BDE0FE',  # light blue 🩵
    '#CDB4DB',  # pastel lavender 💜
    '#FFC8DD',  # pastel pink 🌸
    '#FFAFCC',  # soft pink 🩷
    '#FFD6A5'   # pastel peach 🍑
]

ax = sns.barplot(
    x=intent_default_rate.index,
    y=intent_default_rate.values,
    hue=intent_default_rate.index,
    palette=colors[:len(intent_default_rate)],
    legend=False
)

plt.title(
    'Default Rate by Loan Purpose',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Purpose', fontsize=12)
plt.ylabel('Default Rate (%)', fontsize=12)

plt.xticks(rotation=45)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=9,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
previous_default_rate = (
    df.groupby('cb_person_default_on_file')['loan_status']
    .mean()
    .mul(100)
)

print(previous_default_rate)

In [ ]:
plt.figure(figsize=(8, 5))

colors = [
    '#A8DADC',  # pastel blue
    '#FFAFCC'   # soft pink
]

ax = sns.barplot(
    x=previous_default_rate.index,
    y=previous_default_rate.values,
    hue=previous_default_rate.index,
    palette=colors[:len(previous_default_rate)],
    legend=False
)

plt.title(
    'Current Loan Default Rate by Previous Default History',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Previous Default on File', fontsize=12)
plt.ylabel('Current Default Rate (%)', fontsize=12)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=10,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

## 6. Multivariate Analysis

In [ ]:
numerical_columns = [
    'person_age',
    'person_income',
    'person_emp_length',
    'loan_amnt',
    'loan_int_rate',
    'loan_percent_income',
    'cb_person_cred_hist_length',
    'loan_status'
]

In [ ]:
correlation_matrix = df[numerical_columns].corr()

correlation_matrix

In [ ]:
plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='PuRd',          # cute pastel pink/purple gradient
    linewidths=0.5,
    linecolor='white',
    annot_kws={'fontsize': 10}
)

plt.title(
    'Correlation Heatmap',
    fontsize=16,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
df[numerical_columns].corr()['loan_status'].sort_values(
    ascending=False
)

In [ ]:
df['income_group'] = pd.qcut(
    df['person_income'],
    q=4,
    labels=['Low', 'Medium', 'High', 'Very High']
)

In [ ]:
df['income_group'].value_counts()

In [ ]:
income_default_rate = (
    df.groupby('income_group', observed=True)['loan_status']
    .mean()
    .mul(100)
)

income_default_rate

In [ ]:
plt.figure(figsize=(10, 5))

colors = [
    '#A8DADC',  # pastel blue
    '#BDE0FE',  # light blue
    '#CDB4DB',  # lavender
    '#FFC8DD',  # pastel pink
    '#FFAFCC',  # soft pink
    '#FFD6A5',  # peach
    '#CAFFBF'   # pastel green
]

ax = sns.barplot(
    x=income_default_rate.index,
    y=income_default_rate.values,
    hue=income_default_rate.index,
    palette=colors[:len(income_default_rate)],
    legend=False
)

plt.title(
    'Default Rate by Income Group',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Income Group', fontsize=12)
plt.ylabel('Default Rate (%)', fontsize=12)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=10,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df['age_group'] = pd.cut(
    df['person_age'],
    bins=[17, 25, 35, 50, 100],
    labels=['18-25', '26-35', '36-50', '51+']
)

In [ ]:
df['age_group'].value_counts().sort_index()

In [ ]:
age_default_rate = (
    df.groupby('age_group', observed=True)['loan_status']
    .mean()
    .mul(100)
)

age_default_rate

In [ ]:
plt.figure(figsize=(10, 5))

colors = [
    '#A8DADC',  # pastel blue
    '#BDE0FE',  # light blue
    '#CDB4DB',  # lavender
    '#FFC8DD',  # pastel pink
    '#FFAFCC',  # soft pink
    '#FFD6A5',  # peach
    '#CAFFBF'   # pastel green
]

ax = sns.barplot(
    x=age_default_rate.index,
    y=age_default_rate.values,
    hue=age_default_rate.index,
    palette=colors[:len(age_default_rate)],
    legend=False
)

plt.title(
    'Default Rate by Age Group',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Age Group', fontsize=12)
plt.ylabel('Default Rate (%)', fontsize=12)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=10,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df['loan_income_group'] = pd.qcut(
    df['loan_percent_income'],
    q=4,
    labels=['Low', 'Medium', 'High', 'Very High']
)

In [ ]:
loan_income_default_rate = (
    df.groupby('loan_income_group', observed=True)['loan_status']
    .mean()
    .mul(100)
)

loan_income_default_rate

In [ ]:
plt.figure(figsize=(10, 5))

colors = [
    '#A8DADC',  # pastel blue
    '#BDE0FE',  # light blue
    '#CDB4DB',  # lavender
    '#FFC8DD',  # pastel pink
    '#FFAFCC',  # soft pink
    '#FFD6A5',  # peach
    '#CAFFBF'   # pastel green
]

ax = sns.barplot(
    x=loan_income_default_rate.index,
    y=loan_income_default_rate.values,
    hue=loan_income_default_rate.index,
    palette=colors[:len(loan_income_default_rate)],
    legend=False
)

plt.title(
    'Default Rate by Loan-to-Income Group',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan-to-Income Group', fontsize=12)
plt.ylabel('Default Rate (%)', fontsize=12)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=10,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('loan_grade')['loan_int_rate'].agg(
    ['mean', 'median', 'min', 'max']
).round(2)

In [ ]:
plt.figure(figsize=(10, 5))

grade_order = sorted(df['loan_grade'].unique())

colors = [
    '#A8DADC',  # pastel blue
    '#BDE0FE',  # light blue
    '#CDB4DB',  # lavender
    '#FFC8DD',  # pastel pink
    '#FFAFCC',  # soft pink
    '#FFD6A5',  # peach
    '#CAFFBF'   # pastel green
]

sns.boxplot(
    data=df,
    x='loan_grade',
    y='loan_int_rate',
    order=grade_order,
    hue='loan_grade',
    palette=colors[:len(grade_order)],
    legend=False,
    width=0.6,
    linewidth=1.5,
    fliersize=4
)

plt.title(
    'Interest Rate Distribution by Loan Grade',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Grade', fontsize=12)
plt.ylabel('Interest Rate (%)', fontsize=12)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
grade_analysis = (
    df.groupby('loan_grade')
    .agg(
        total_loans=('loan_status', 'count'),
        defaults=('loan_status', 'sum'),
        default_rate=('loan_status', 'mean')
    )
)

grade_analysis['default_rate'] = (
    grade_analysis['default_rate'] * 100
).round(2)

grade_analysis

In [ ]:
plt.figure(figsize=(10, 5))

colors = [
    '#A8DADC',  # pastel blue
    '#BDE0FE',  # light blue
    '#CDB4DB',  # lavender
    '#FFC8DD',  # pastel pink
    '#FFAFCC',  # soft pink
    '#FFD6A5',  # peach
    '#CAFFBF'   # pastel green
]

ax = sns.barplot(
    x=grade_analysis.index,
    y=grade_analysis['default_rate'],
    hue=grade_analysis.index,
    palette=colors[:len(grade_analysis)],
    legend=False
)

plt.title(
    'Default Rate by Loan Grade',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan Grade', fontsize=12)
plt.ylabel('Default Rate (%)', fontsize=12)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=10,
        padding=3
    )

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
grade_interest_default = (
    df.groupby('loan_grade')
    .agg(
        avg_interest_rate=('loan_int_rate', 'mean'),
        default_rate=('loan_status', 'mean'),
        loan_count=('loan_status', 'count')
    )
)

grade_interest_default['avg_interest_rate'] = (
    grade_interest_default['avg_interest_rate'].round(2)
)

grade_interest_default['default_rate'] = (
    grade_interest_default['default_rate'] * 100
).round(2)

grade_interest_default

In [ ]:
pd.crosstab(
    df['loan_grade'],
    df['cb_person_default_on_file'],
    normalize='index'
).round(3) * 100

In [ ]:
grade_income_default = pd.pivot_table(
    df,
    values='loan_status',
    index='loan_grade',
    columns='loan_income_group',
    aggfunc='mean'
) * 100

grade_income_default.round(2)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Cute pastel color gradient
pastel_cmap = LinearSegmentedColormap.from_list(
    'pastel_loan',
    [
        '#BDE0FE',  # pastel blue
        '#CDB4DB',  # lavender
        '#FFC8DD',  # pastel pink
        '#FFAFCC'   # soft pink
    ]
)

plt.figure(figsize=(10, 6))

sns.heatmap(
    grade_income_default,
    annot=True,
    fmt='.2f',
    cmap=pastel_cmap,
    linewidths=0.5,
    linecolor='white',
    annot_kws={'fontsize': 10}
)

plt.title(
    'Default Rate by Loan Grade and Loan-to-Income Group',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Loan-to-Income Group', fontsize=12)
plt.ylabel('Loan Grade', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
previous_default_analysis = (
    df.groupby('cb_person_default_on_file')['loan_status']
    .agg(
        total_applicants='count',
        defaults='sum',
        default_rate='mean'
    )
)

previous_default_analysis['default_rate'] = (
    previous_default_analysis['default_rate'] * 100
).round(2)

previous_default_analysis

## 7. Statistical Analysis

In [ ]:
default_history_table = pd.crosstab(
    df['cb_person_default_on_file'],
    df['loan_status']
)

default_history_table

In [ ]:
from scipy.stats import chi2_contingency

In [ ]:
chi2, p_value, dof, expected = chi2_contingency(
    default_history_table
)

print("Chi-Square Statistic:", chi2)
print("P-value:", p_value)

In [ ]:
total_applicants = len(df)
total_defaults = df['loan_status'].sum()

default_rate = (total_defaults / total_applicants) * 100

print("Total Applicants:", total_applicants)
print("Total Defaults:", total_defaults)
print(f"Overall Default Rate: {default_rate:.2f}%")

In [ ]:
home_default = (
    df.groupby('person_home_ownership')['loan_status']
    .agg(['count', 'sum', 'mean'])
)

home_default.columns = [
    'Total Applicants',
    'Total Defaults',
    'Default Rate'
]

home_default['Default Rate'] = (
    home_default['Default Rate'] * 100
).round(2)

home_default.sort_values('Default Rate', ascending=False)

In [ ]:
intent_default = (
    df.groupby('loan_intent')['loan_status']
    .agg(['count', 'sum', 'mean'])
)

intent_default.columns = [
    'Total Applicants',
    'Total Defaults',
    'Default Rate'
]

intent_default['Default Rate'] = (
    intent_default['Default Rate'] * 100
).round(2)

intent_default.sort_values('Default Rate', ascending=False)

In [ ]:
grade_default = (
    df.groupby('loan_grade')['loan_status']
    .agg(['count', 'sum', 'mean'])
)

grade_default.columns = [
    'Total Loans',
    'Total Defaults',
    'Default Rate'
]

grade_default['Default Rate'] = (
    grade_default['Default Rate'] * 100
).round(2)

grade_default.sort_index()

In [ ]:
history_default = (
    df.groupby('cb_person_default_on_file')['loan_status']
    .agg(['count', 'sum', 'mean'])
)

history_default.columns = [
    'Total Applicants',
    'Current Defaults',
    'Current Default Rate'
]

history_default['Current Default Rate'] = (
    history_default['Current Default Rate'] * 100
).round(2)

history_default

In [ ]:
risk_summary = df.groupby('loan_grade').agg(
    applicants=('loan_status', 'count'),
    default_rate=('loan_status', 'mean'),
    avg_income=('person_income', 'mean'),
    avg_loan_amount=('loan_amnt', 'mean'),
    avg_interest_rate=('loan_int_rate', 'mean'),
    avg_loan_income_ratio=('loan_percent_income', 'mean')
)

risk_summary['default_rate'] = (
    risk_summary['default_rate'] * 100
).round(2)

risk_summary = risk_summary.round(2)

risk_summary

In [ ]:
df.to_csv(
    "credit_risk_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

## 8. Key Insights & Business Takeaways

In [ ]:
highest_grade = grade_default['Default Rate'].idxmax()
highest_grade_rate = grade_default['Default Rate'].max()

lowest_grade = grade_default['Default Rate'].idxmin()
lowest_grade_rate = grade_default['Default Rate'].min()

print(
    f"Highest default rate: Grade {highest_grade} "
    f"({highest_grade_rate:.2f}%)"
)

print(
    f"Lowest default rate: Grade {lowest_grade} "
    f"({lowest_grade_rate:.2f}%)"
)

In [ ]:
# Highest-risk loan purpose

highest_intent = intent_default['Default Rate'].idxmax()
highest_intent_rate = intent_default['Default Rate'].max()

print(
    f"Loan intent with highest default rate: "
    f"{highest_intent} ({highest_intent_rate:.2f}%)"
)

In [ ]:
# Highest-risk home ownership category

highest_home = home_default['Default Rate'].idxmax()
highest_home_rate = home_default['Default Rate'].max()

print(
    f"Home ownership category with highest default rate: "
    f"{highest_home} ({highest_home_rate:.2f}%)"
)